In [2]:
import os
import glob
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy.spatial.distance import pdist, squareform

from models.resnet_embedder import ResNetEmbedder
from models.clip_embedder import CLIPEmbedder
from models.vlm_embedder import VLMEmbedder

RuntimeError: operator torchvision::nms does not exist

In [ ]:
STIMULI_DIR = "stimuli/gw"
VALID_EXTS = (".jpg", ".jpeg", ".png", ".bmp")

stimulus_paths = sorted(
    p for p in glob.glob(os.path.join(STIMULI_DIR, "*"))
    if p.lower().endswith(VALID_EXTS)
)
stimulus_names = [os.path.splitext(os.path.basename(p))[0] for p in stimulus_paths]
stimuli = {name: Image.open(p).convert("RGB") for name, p in zip(stimulus_names, stimulus_paths)}

print(f"Loaded {len(stimuli)} stimuli from {STIMULI_DIR}:")
print(stimulus_names)

In [ ]:
fig, axes = plt.subplots(1, len(stimuli), figsize=(3 * len(stimuli), 3))
if len(stimuli) == 1:
    axes = [axes]
for ax, name in zip(axes, stimulus_names):
    ax.imshow(stimuli[name])
    ax.set_title(name, fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
print("Loading ResNet...")
resnet_embedder = ResNetEmbedder()

print("Loading CLIP...")
clip_embedder = CLIPEmbedder()

print("Loading VLM...")
vlm_embedder = VLMEmbedder()

In [ ]:
def to_numpy(embedding):
    """Flatten a torch.Tensor or np.ndarray embedding into a 1D numpy vector."""
    if hasattr(embedding, "detach"):
        embedding = embedding.detach().cpu().numpy()
    return np.asarray(embedding).reshape(-1)

In [ ]:
methods = {
    "ResNet": resnet_embedder,
    "CLIP": clip_embedder,
    "VLM": vlm_embedder,
}

embeddings = {method_name: {} for method_name in methods}

for method_name, embedder in methods.items():
    print(f"Embedding stimuli with {method_name}...")
    for name in stimulus_names:
        vec = embedder.get_embedding(stimuli[name])
        embeddings[method_name][name] = to_numpy(vec)

In [ ]:
FEATURE_PROMPT = "Describe this image."

embeddings["VLM-text"] = {}
for name in stimulus_names:
    vec = vlm_embedder.get_text_embedding(stimuli[name], FEATURE_PROMPT)
    embeddings["VLM-text"][name] = to_numpy(vec)

In [ ]:
def compute_rdm(embeddings_dict, names, metric="correlation"):
    """
    Build a representational dissimilarity matrix (RDM).

    Args:
        embeddings_dict: dict mapping stimulus name -> 1D embedding vector
        names: ordered list of stimulus names (defines row/col order)
        metric: distance metric passed to scipy.spatial.distance.pdist
            (e.g. "correlation", "cosine", "euclidean")

    Returns:
        np.ndarray of shape (n, n) — the RDM
    """
    matrix = np.stack([embeddings_dict[name] for name in names])
    distances = pdist(matrix, metric=metric)
    return squareform(distances)

In [ ]:
RDM_METRIC = "correlation"  # also try "cosine" or "euclidean"

rdms = {
    method_name: compute_rdm(embeddings[method_name], stimulus_names, metric=RDM_METRIC)
    for method_name in embeddings
}

In [ ]:
fig, axes = plt.subplots(1, len(rdms), figsize=(6 * len(rdms), 5))
if len(rdms) == 1:
    axes = [axes]

for ax, (method_name, rdm) in zip(axes, rdms.items()):
    im = ax.imshow(rdm, cmap="viridis")
    ax.set_xticks(range(len(stimulus_names)))
    ax.set_yticks(range(len(stimulus_names)))
    ax.set_xticklabels(stimulus_names, rotation=90, fontsize=8)
    ax.set_yticklabels(stimulus_names, fontsize=8)
    ax.set_title(f"{method_name} RDM", fontsize=12, fontweight="bold")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.show()